In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

from collections import Counter

In [ ]:
df = pd.read_csv("../data/split/train_reviews.csv")

In [ ]:
pd.set_option("display.max_columns", None)
df.head(2)

In [ ]:
df["LABEL-simple_rating"].value_counts()

### Ile znajduje się w zbiorze cech kategorycznych, a ile numerycznych?

In [ ]:
feature_cols = [c for c in df.columns if c != "LABEL-simple_rating"]
numeric_cols = (
    df[feature_cols].select_dtypes(include=["number", "bool"]).columns.tolist()
)
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

summary = pd.DataFrame(
    {
        "feature_type": ["numeric", "categorical"],
        "count": [len(numeric_cols), len(categorical_cols)],
        "columns_preview": [
            ", ".join(numeric_cols[:8]),
            ", ".join(categorical_cols[:8]),
        ],
    }
)
summary

### Missing Values

In [ ]:
df.isna().sum()

### Czy któreś z cech są skorelowane? Co z tego może wynikać?

In [ ]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns
correlation_matrix = df[numerical_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

- price_usd i price_usd_right korelacja 1.0 - oznacza to, że te dwie cechy są identyczne - jedna do usunięcia
- loves_count i reviews - popularne cechy, które mogą być skorelowane z popularnością produktu
- child_count i loves_count / child_count i reviews - produkty posiadajace kilka wersji mogą być bardziej popularne


### Czy któraś z cech koreluje ze zmienną wyjściową? Jeśli tak - która? Czy któraś nie koreluje?


- is_recommended - silnie skorelowane ze zmienna wyjściową (0.75)
- rating - umiarkowanie skorelowane (0.24)
- total_neg_feedback_count - umiarkowanie skorelowane (-0.15)


### Czy któreś ze słów wydają się dominować w zbiorze?

In [ ]:
texts = df["review_text"].dropna().str.lower()

texts = texts.apply(lambda x: re.sub(r"[^\w\s]", "", x))

words = " ".join(texts).split()

stopwords = set(
    [
        "the",
        "and",
        "is",
        "in",
        "it",
        "to",
        "of",
        "for",
        "on",
        "this",
        "that",
        "was",
        "with",
        "as",
        "but",
        "are",
        "a",
        "so",
        "has",
    ]
)
words = [w for w in words if w not in stopwords]

word_counts = Counter(words)

top_words = word_counts.most_common(20)
words, counts = zip(*top_words)

plt.figure()
plt.bar(words, counts)
plt.xticks(rotation=45)
plt.title("Top 20 najczęstszych słów w review_text")
plt.xlabel("Słowa")
plt.ylabel("Liczba wystąpień")
plt.tight_layout()
plt.show()

czy najpopularniejsze słowa różnią się znacząco pomiędzy klasami? Czy potrafisz wyróżnić słowa mogące wpływać w znaczym stopniu na sentyment?

In [ ]:
target = "LABEL-simple_rating"


def get_top_words(series, n=15):
    texts = series.dropna().str.lower()
    texts = texts.apply(lambda x: re.sub(r"[^\w\s]", "", x))

    words = " ".join(texts).split()

    stopwords = set(
        [
            "the",
            "and",
            "is",
            "in",
            "it",
            "to",
            "of",
            "for",
            "on",
            "this",
            "that",
            "was",
            "with",
            "as",
            "but",
            "are",
            "a",
            "so",
            "has",
            "i",
            "my",
            "skin",
            "product",
            "have",
        ]
    )

    words = [w for w in words if w not in stopwords]

    return Counter(words).most_common(n)


classes = df[target].dropna().unique()

plt.figure(figsize=(12, 5))

for i, cls in enumerate(classes):
    top_words = get_top_words(df[df[target] == cls]["review_text"], 15)
    words, counts = zip(*top_words)

    plt.subplot(1, len(classes), i + 1)
    plt.bar(words, counts)
    plt.xticks(rotation=45)
    plt.title(f"Top words: {cls}")

plt.tight_layout()
plt.show()

- większosc slow jest neutralna i pojawia się w kazdej klasie
- slowa 'love', 'very', 'good' pojawiaja sie czesto w klasie 1 (pozytywne recenzje)
- negacje zaczynaja sie pojawiac coraz czesciej odpowiednio w klasie 2 i 3
- slowa out, face, not just moga sugerowac pewne problemy techniczne z produktem w stylu: produkt po prostu NIE dziala

### jaka jest charakterystyka tekstu (np. długość, czystość)?


In [ ]:
target = "LABEL-simple_rating"

df = df.copy()
texts = df["review_text"].fillna("")

df["char_len"] = texts.str.len()
df["word_len"] = texts.str.split().apply(len)
df["avg_word_len"] = texts.apply(
    lambda x: (
        sum(len(w) for w in x.split()) / len(x.split()) if len(x.split()) > 0 else 0
    )
)
df["special_chars"] = texts.apply(lambda x: len(re.findall(r"[^\w\s]", x)))
df["digits"] = texts.apply(lambda x: len(re.findall(r"\d", x)))

df["special_ratio"] = df["special_chars"] / df["char_len"].replace(0, 1)
df["digit_ratio"] = df["digits"] / df["char_len"].replace(0, 1)

In [ ]:
def plot_feature_by_class(df, feature, target):
    classes = df[target].dropna().unique()

    plt.figure(figsize=(12, 5))
    medians = []
    for i, cls in enumerate(classes):
        plt.subplot(1, len(classes), i + 1)

        data = df[df[target] == cls][feature].dropna()
        plt.hist(data, bins=40)
        plt.axvline(data.mean(), color="red", linestyle="dashed", linewidth=1)
        plt.title(f"{feature} | {cls}")
        plt.xlabel(feature)
        plt.ylabel("count")

        medians.append(data.median())
    print(
        f"Medians for {feature} by class: {', '.join([f'{cls}: {median:.2f}' for cls, median in zip(classes, medians)])}"
    )
    plt.tight_layout()
    plt.show()

In [ ]:
plot_feature_by_class(df, "word_len", target)
plot_feature_by_class(df, "char_len", target)
plot_feature_by_class(df, "avg_word_len", target)
plot_feature_by_class(df, "special_ratio", target)
plot_feature_by_class(df, "digit_ratio", target)

- nieznacznie dłuzsze recenzje pojawiaja sie w klasie 2, co ma swoje odzwierciedlenie w liczbe slow i liczba znakow per review

### Pytanie dodatkowe 1: Czy recenzje zawierają ekstremalne emocje (!!!, CAPS)?

In [ ]:
texts = df["review_text"].fillna("")

df["exclamations"] = texts.apply(lambda x: x.count("!"))

df["capital_ratio"] = texts.apply(
    lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0
)

In [ ]:
plt.figure()

for cls in df[target].dropna().unique():
    data = df[df[target] == cls]["exclamations"]
    plt.hist(data, bins=50, alpha=0.5, label=str(cls))

plt.legend()
plt.title("Użycie wykrzykników vs klasa")
plt.xlabel("exclamations")
plt.ylabel("count")
plt.show()

In [ ]:
plt.figure()

for cls in df[target].dropna().unique():
    data = df[df[target] == cls]["capital_ratio"]
    plt.hist(data, bins=50, alpha=0.5, label=str(cls))

plt.legend()
plt.title("Użycie wielkich liter vs klasa")
plt.xlabel("capital_ratio")
plt.ylabel("count")
plt.show()

### Pytanie dodatkowe 2: Czy recenzje zawierają dużo powtórzeń?

In [ ]:
def repetition_ratio(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return len(words) / len(set(words))


df["repetition_ratio"] = (
    df["review_text"].fillna("").str.lower().apply(repetition_ratio)
)

In [ ]:
df["repetition_ratio"] = (
    df["review_text"].fillna("").str.lower().apply(repetition_ratio)
)

plt.figure()

for cls in df[target].dropna().unique():
    data = df[df[target] == cls]["repetition_ratio"]
    plt.hist(data, bins=50, alpha=0.5, label=str(cls))

plt.legend()
plt.title("Powtarzalność słów vs klasa")
plt.xlabel("repetition_ratio")
plt.ylabel("count")
plt.show()

### Czy zbiór danych wydaje się być wystarczająco informacyjny by rozwiązać zadanie analizy sentymentu?


**1. Korelacje i selekcja cech**
- Zmienne `price_usd` i `price_usd_right` – są identyczne i jedną z nich należy usunąć ze zbioru.
- Zmienna docelowa (`LABEL-simple_rating`) wykazuje najsilniejszą korelację ze zmienną `is_recommended` (0.75) - w realnym scenariuszu rekomendacja produktu (gdy uzytkownik pisze recenzje) rekomendacja powstaje w tym samym momencie co ocena, w eksperymentach bedzie uwzgledniane modelowanie bez tego atrybutu. Umiarkowanie koreluje również z `rating` oraz ujemnie z `total_neg_feedback_count`
- Cechy `child_count` oraz `loves_count` i `reviews` są powiązane, co wskazuje, że produkty posiadające kilka wersji cieszą się większą popularnością.

**2. Słownictwo i analiza sentymentu**
- Większość słów w recenzjach jest neutralna i powtarza się w każdej z klas.
- **Klasa 1 (pozytywne):** Wyraźnie dominują słowa *love*, *very*, *good*.
- **Klasy 2 i 3 (neutralne/negatywne):** Występuje w nich znacznie więcej negacji (np. *not*). Obecność słów takich jak *out*, *face*, *not*, *just* sugeruje powtarzające się problemy z produktami.

**3. Struktura i długość tekstu**
- Najdłuższe recenzje znajdują się w klasie 2 (mediana: 54 słowa / 285 znaków).
- Recenzje skrajne są krótsze: klasa 3 ma medianę 48 słów (256 znaków), a klasa 1 jest najkrótsza z medianą 46 słów (242 znaki).
- Średnia długość słowa jest podobna we wszystkich klasach i wynosi od 4.25 do 4.33 znaku.
- Udział znaków specjalnych we wszystkich klasach wynosi dokładnie tyle samo (mediana: 0.03), a cyfr w tekstach praktycznie nie ma (mediana: 0.00).
